In [0]:
# Questo notebook analizza l'andamento mensile della produzione rinnovabile in Italia.
#
# L'obiettivo è capire come cambia la produzione durante l'anno,
# osservando prima il contributo combinato di Solare ed Eolico
# e poi il mix completo composto da Solare, Eolico e Idrico.
#
# Il notebook usa la tabella Gold finale:
# progetto_meteo.gold.dati_studio
#
# Questa tabella contiene già il dato finale unificato tra meteo ed energia,
# aggregato per Area, Data e Ora.


import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots


# Imposto il catalogo e la tabella Gold finale.
catalogo = "progetto_meteo"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"

colore_solare = "#FFC000"
colore_eolico = "#40DD3B"
colore_idrico = "#00B0F0"
colore_media = "#FF0000"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Controllo che la Gold finale sia popolata.
# Se la tabella è vuota, i casi studio non possono essere eseguiti.
righe_gold = spark.table(tabella_dati_studio).count()

if righe_gold == 0:
    raise Exception(
        f"La tabella {tabella_dati_studio} è vuota. "
        "Prima esegui Gold_Dati_Aggregati e Gold_Dati_Studio."
    )

print(f"Tabella pronta: {tabella_dati_studio}")
print(f"Righe disponibili: {righe_gold}")


# Leggo una query direttamente da Databricks.
# Dentro Databricks il notebook può usare Spark senza credenziali locali.
def leggi_databricks(query):

    return spark.sql(query).toPandas()


# Mostro i grafici Plotly dentro Databricks.
# displayHTML rende il grafico più stabile nei notebook Databricks.
def mostra_grafico(fig):

    displayHTML(
        fig.to_html(
            include_plotlyjs=True,
            full_html=False,
            config={"responsive": True}
        )
    )


# Converto colonne pandas in numerico quando serve.
# Questo evita problemi di visualizzazione con Plotly.
def converti_colonne_numeriche(df, colonne):

    for colonna in colonne:
        if colonna in df.columns:
            df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

    return df

In [0]:
# Studio 1A - Produzione mensile Solare + Eolico in Italia
#
# Questa prima analisi isola le due fonti rinnovabili più direttamente legate
# alle condizioni atmosferiche immediate:
# - Solare, legato alla disponibilità di luce e alla stagionalità;
# - Eolico, legato alla presenza e intensità del vento.
#
# L'obiettivo è osservare quanto producono insieme mese per mese
# e capire se durante l'anno esistono periodi di maggiore o minore disponibilità.
#
# Possibile utilizzo:
# questa lettura può supportare analisi di stagionalità energetica,
# valutazioni sul bilanciamento della produzione rinnovabile
# e confronti preliminari tra mesi più favorevoli o più deboli.
#
# In un contesto economico può aiutare a ragionare su esposizione stagionale,
# fabbisogni di backup, pianificazione della capacità o dipendenza da fonti variabili.


# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo la produzione mensile aggregata dalla Gold finale.
query_studio_1 = f"""
SELECT
    date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM') AS Mese,
    CAST(ROUND(SUM(Solare) / 1000000, 2) AS DOUBLE) AS Solare,
    CAST(ROUND(SUM(Eolico) / 1000000, 2) AS DOUBLE) AS Eolico,
    CAST(ROUND(SUM(Solare + Eolico) / 1000000, 2) AS DOUBLE) AS Totale_Rinnovabile
FROM {tabella_dati_studio}
WHERE year(to_date(Data, 'dd/MM/yyyy')) = {anno_studio}
GROUP BY date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM')
ORDER BY Mese
"""

df_studio_1 = leggi_databricks(query_studio_1)


# Converto le colonne numeriche per evitare problemi con Plotly.
df_studio_1 = converti_colonne_numeriche(
    df_studio_1,
    [
        "Solare",
        "Eolico",
        "Totale_Rinnovabile"
    ]
)


# Controllo i dati letti.
display(df_studio_1)

print("Righe lette:", len(df_studio_1))


# Calcolo media mensile e totale annuo.
# Questi valori servono per dare un riferimento sintetico al grafico.
media_mensile = df_studio_1["Totale_Rinnovabile"].mean()
totale_annuo = df_studio_1["Totale_Rinnovabile"].sum()


# Creo il grafico stacked mensile.
fig = go.Figure()


# Aggiungo la produzione solare mensile.
fig.add_trace(
    go.Bar(
        x=df_studio_1["Mese"],
        y=df_studio_1["Solare"],
        name="Solare",
        marker_color=colore_solare,
        hovertemplate="Mese: %{x}<br>Solare: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo la produzione eolica mensile.
fig.add_trace(
    go.Bar(
        x=df_studio_1["Mese"],
        y=df_studio_1["Eolico"],
        name="Eolico",
        marker_color=colore_eolico,
        hovertemplate="Mese: %{x}<br>Eolico: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo il totale mensile sopra ogni barra.
fig.add_trace(
    go.Scatter(
        x=df_studio_1["Mese"],
        y=df_studio_1["Totale_Rinnovabile"],
        mode="text",
        text=df_studio_1["Totale_Rinnovabile"].round(1).astype(str) + " TWh",
        textposition="top center",
        showlegend=False,
        hoverinfo="skip"
    )
)


# Aggiungo la media mensile come informazione di sintesi in legenda.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name=f"Media mensile = {media_mensile:.1f} TWh",
        marker=dict(color=colore_media, size=10),
        hoverinfo="skip"
    )
)


# Sistemo il layout del grafico.
fig.update_layout(
    title=f"Produzione mensile Solare + Eolico in Italia - {anno_studio}<br><sup>Totale annuo: {totale_annuo:.1f} TWh</sup>",
    xaxis_title="Mese",
    yaxis_title="Produzione combinata (TWh)",
    barmode="stack",
    height=650,
    title_x=0.5,
    legend_title="Fonte",
    hovermode="x unified"
)

fig.update_yaxes(showgrid=True)
fig.update_xaxes(tickangle=-45)


# Mostro il grafico.
mostra_grafico(fig)

In [0]:
# Studio 1B - Produzione mensile rinnovabile completa in Italia
#
# Questa seconda analisi amplia la lettura aggiungendo anche l'Idrico.
# In questo modo il grafico non mostra solo le fonti più intermittenti,
# ma una visione più completa del mix rinnovabile mensile.
#
# L'obiettivo è capire quanto cambia il peso delle tre componenti
# nel corso dell'anno e osservare se il contributo idrico compensa,
# rafforza o accompagna l'andamento di Solare ed Eolico.
#
# Possibile utilizzo:
# questa analisi può essere utile per confrontare la composizione del mix rinnovabile
# nei diversi mesi e per supportare valutazioni su stabilità, diversificazione
# e dipendenza da specifiche fonti energetiche.
#
# In ambito economico o di investimento può aiutare a leggere quali mesi
# mostrano un mix più equilibrato e quali invece dipendono maggiormente
# da una singola fonte.

# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo la produzione mensile aggregata dalla Gold finale.
query_studio_2 = f"""
SELECT
    date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM') AS Mese,
    CAST(ROUND(SUM(Solare) / 1000000, 2) AS DOUBLE) AS Solare,
    CAST(ROUND(SUM(Eolico) / 1000000, 2) AS DOUBLE) AS Eolico,
    CAST(ROUND(SUM(Idrico) / 1000000, 2) AS DOUBLE) AS Idrico,
    CAST(
        ROUND(
            (
                SUM(Solare) +
                SUM(Eolico) +
                SUM(Idrico)
            ) / 1000000,
            2
        ) AS DOUBLE
    ) AS Totale_Rinnovabile
FROM {tabella_dati_studio}
WHERE year(to_date(Data, 'dd/MM/yyyy')) = {anno_studio}
GROUP BY date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM')
ORDER BY Mese
"""

df_studio_2 = leggi_databricks(query_studio_2)


# Converto le colonne numeriche per evitare problemi con Plotly.
df_studio_2 = converti_colonne_numeriche(
    df_studio_2,
    [
        "Solare",
        "Eolico",
        "Idrico",
        "Totale_Rinnovabile"
    ]
)


# Controllo i dati letti.
display(df_studio_2)

print("Righe lette:", len(df_studio_2))


# Calcolo media mensile e totale annuo.
# Questi valori aiutano a leggere il grafico in modo più immediato.
media_mensile = df_studio_2["Totale_Rinnovabile"].mean()
totale_annuo = df_studio_2["Totale_Rinnovabile"].sum()


# Creo il grafico stacked mensile.
fig = go.Figure()


# Aggiungo la produzione solare mensile.
fig.add_trace(
    go.Bar(
        x=df_studio_2["Mese"],
        y=df_studio_2["Solare"],
        name="Solare",
        marker_color=colore_solare,
        hovertemplate="Mese: %{x}<br>Solare: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo la produzione eolica mensile.
fig.add_trace(
    go.Bar(
        x=df_studio_2["Mese"],
        y=df_studio_2["Eolico"],
        name="Eolico",
        marker_color=colore_eolico,
        hovertemplate="Mese: %{x}<br>Eolico: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo la produzione idrica mensile.
fig.add_trace(
    go.Bar(
        x=df_studio_2["Mese"],
        y=df_studio_2["Idrico"],
        name="Idrico",
        marker_color=colore_idrico,
        hovertemplate="Mese: %{x}<br>Idrico: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo il totale mensile sopra ogni barra.
fig.add_trace(
    go.Scatter(
        x=df_studio_2["Mese"],
        y=df_studio_2["Totale_Rinnovabile"],
        mode="text",
        text=df_studio_2["Totale_Rinnovabile"].round(1).astype(str) + " TWh",
        textposition="top center",
        showlegend=False,
        hoverinfo="skip"
    )
)


# Aggiungo la media mensile come informazione di sintesi in legenda.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name=f"Media mensile = {media_mensile:.1f} TWh",
        marker=dict(color=colore_media, size=10),
        hoverinfo="skip"
    )
)


# Sistemo il layout del grafico.
fig.update_layout(
    title=f"Produzione mensile rinnovabile in Italia - {anno_studio}<br><sup>Totale annuo: {totale_annuo:.1f} TWh</sup>",
    xaxis_title="Mese",
    yaxis_title="Produzione rinnovabile (TWh)",
    barmode="stack",
    height=650,
    title_x=0.5,
    legend_title="Fonte",
    hovermode="x unified"
)

fig.update_yaxes(showgrid=True)
fig.update_xaxes(tickangle=-45)


# Mostro il grafico.
mostra_grafico(fig)